In [2]:
import duckdb
from pathlib import Path

def logical_compare_parquet(dir1, dir2):
    dir1 = str(Path(dir1))
    dir2 = str(Path(dir2))
    con = duckdb.connect()

    # List .parquet files in both dirs
    names1 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir1}/*.parquet')").fetchall()}
    names2 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir2}/*.parquet')").fetchall()}
    common = sorted(names1 & names2)

    only1 = sorted(names1 - names2)
    only2 = sorted(names2 - names1)
    print(f"Only in {dir1}: {only1}")
    print(f"Only in {dir2}: {only2}")

    all_ok = True
    for name in common:
        path1 = f"{dir1}/{name}"
        path2 = f"{dir2}/{name}"

        # Read schemas
        s1 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path1}')").fetchdf()
        s2 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path2}')").fetchdf()
        schema_equal = s1[['column_name','column_type']].equals(s2[['column_name','column_type']])

        # Row counts
        n1 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path1}')").fetchone()[0]
        n2 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path2}')").fetchone()[0]

        # Set difference both ways (assumes rows are comparable; if there’s no stable key,
        # this treats the entire row as a tuple)
        diff12 = con.execute(
            f"SELECT * FROM parquet_scan('{path1}') EXCEPT ALL SELECT * FROM parquet_scan('{path2}')"
        ).fetchone()
        diff21 = con.execute(
            f"SELECT * FROM parquet_scan('{path2}') EXCEPT ALL SELECT * FROM parquet_scan('{path1}')"
        ).fetchone()

        # If EXCEPT returns at least one row, DuckDB returns a row; we only need to know emptiness.
        has_diff12 = diff12 is not None
        has_diff21 = diff21 is not None

        if schema_equal and n1 == n2 and not has_diff12 and not has_diff21:
            print(f"[OK]   {name}: same schema, same rows ({n1} rows).")
        else:
            all_ok = False
            print(f"[DIFF] {name}: schema_equal={schema_equal}, rowsA={n1}, rowsB={n2}, "
                  f"A\\B_diff={has_diff12}, B\\A_diff={has_diff21}")

    if all_ok and not only1 and not only2:
        print("✅ All matching files have identical table contents.")
    else:
        print("❗Content differences found (see above).")

# Example:
path_a = "/shared/share_mala/zj2398/mimic/models/motor_8k/labels"
path_b = "/shared/share_mala/zj2398/mimic/models/mtpp_8k/labels"
logical_compare_parquet(path_a,path_b)


Only in /shared/share_mala/zj2398/mimic/models/motor_8k/labels: []
Only in /shared/share_mala/zj2398/mimic/models/mtpp_8k/labels: []
[OK]   celiac.parquet: same schema, same rows (214715 rows).
[OK]   in_hospital_mortality.parquet: same schema, same rows (331592 rows).
[OK]   long_los.parquet: same schema, same rows (331593 rows).
[OK]   masld.parquet: same schema, same rows (209133 rows).
[OK]   readmission.parquet: same schema, same rows (418850 rows).
[OK]   stroke.parquet: same schema, same rows (314140 rows).
✅ All matching files have identical table contents.


In [2]:
import pickle
import pandas as pd
task = "readmission"
features_label_input_pkl = f"/shared/share_mala/zj2398/mimic/task/patient_outcome_task/motor/features/{task}_motor.pkl"
label_path = f"/shared/share_mala/zj2398/mimic/task/patient_outcome_task/motor/labels/{task}.parquet"
with open(features_label_input_pkl,'rb') as file:
    features_label = pickle.load(file)
label = pd.read_parquet(label_path)
print(features_label)
print(features_label['subject_ids'].shape,features_label['feature_times'].shape)
print()
print(label)
# features_label = features_label.sort("subject_id", "prediction_time")

{'subject_ids': array([17558257, 17558257, 11347735, ..., 13537167, 13537167, 13182648]), 'feature_times': array(['2112-11-28T11:30:00', '2112-12-26T16:55:00',
       '2156-04-17T14:30:00', ..., '2171-12-17T15:30:00',
       '2171-12-27T16:00:00', '2160-01-04T15:13:00'],
      dtype='datetime64[s]'), 'features': array([[-1.0565158 , -0.4983525 ,  0.61977947, ..., -0.40710965,
         0.28894737,  0.6229595 ],
       [-1.827342  , -0.77842057,  0.27558577, ...,  0.12806207,
         0.05674551,  1.1801274 ],
       [ 0.5018504 ,  1.2052355 ,  0.42443913, ..., -0.5173896 ,
        -0.44637284,  0.7874491 ],
       ...,
       [ 0.6891315 ,  0.95598984,  0.69617957, ..., -1.2365667 ,
        -0.13103807,  0.6889306 ],
       [ 0.37021792,  0.8987858 ,  0.07164215, ..., -0.7017305 ,
        -0.22987495,  0.7938947 ],
       [-1.2115433 , -1.5020506 ,  0.8685364 , ..., -0.51140153,
         0.1292431 ,  1.4190873 ]], dtype=float32)}
(703273,) (703273,)

        subject_id     prediction_ti

In [8]:
import pandas as pd
from pathlib import Path
base_dir = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_0.1")
# train_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform/data/train")
# for i in range(0,10):
#     train_cohort = pd.read_parquet(train_path/f"{i}.parquet")
#     print(train_cohort["subject_id"])
# meta_path = base_dir/"metadata"/"subject_splits.parquet"
# meta_path = base_dir/"metadata"/"vocabulary_cache"/"mapping"/"data.parquet"
id_list = []
data_dir = base_dir/"data"
for subdir in data_dir.iterdir():
    if not subdir.is_dir():
        continue
    for f in subdir.glob("*.parquet"):
        raw_file = pd.read_parquet(f)
        id_list.extend(raw_file["subject_id"].unique())
        # break
        # id_list.extend(raw_file)
    # break
print(len(id_list))
# meta_data = pd.read_parquet(meta_path)
# print(meta_data)

690298


In [2]:
from pathlib import Path
import meds_reader
import femr.splits
import pandas as pd

meds_reader_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_meds_reader")
subject_splits_path = meds_reader_path / "metadata/subject_splits.parquet"
code_metadata_path = meds_reader_path / "metadata/codes.parquet"
with meds_reader.SubjectDatabase(str(meds_reader_path), num_threads=16) as database:
    subject_ids = [_ for _ in database]
    subject_splits = pd.read_parquet(subject_splits_path)
    subject_splits = subject_splits[subject_splits.subject_id.isin(subject_ids)]
    train_tuning_split = subject_splits[~subject_splits.split.isin(["held_out"])].subject_id.tolist()
    test_split = subject_splits[subject_splits.split.isin(["held_out"])].subject_id.tolist()
    main_split = femr.splits.SubjectSplit(train_tuning_split, test_split)
    main_split.save_to_csv('/data2/processed_datasets/zj2398/femr/cumc/cumc_split.csv')

/data/anaconda3/envs/motor_chao/lib/python3.11/site-packages/meds_reader/__init__.py:205: UserWarning: Warning: You are using meds_reader multiprocessing within a Jupyter notebook. This does not reliably work and your code might randomly fail.
  warnings.warn("Warning: You are using meds_reader multiprocessing within a Jupyter notebook. This does not reliably work and your code might randomly fail.")


In [3]:
import pandas as pd
from pathlib import Path
held_out_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/AMI")
split = "train.parquet"
sample_parquet = pd.read_parquet(held_out_path/split)
print(sample_parquet)

       subject_id prediction_time  boolean_value  split  tte_label  \
0         7173500      2011-07-31              1  train          1   
1         2229279      2001-10-25              1  train          1   
2        15080688      2020-05-30              1  train          1   
3        13847456      2015-12-25              1  train          1   
4         1746373      2004-04-15              1  train          1   
...           ...             ...            ...    ...        ...   
79340    15116103      2018-03-01              0  train          0   
79341     3951087      2011-12-08              0  train          0   
79342      540844      2016-10-07              0  train          0   
79343     5203792      2017-10-29              0  train          0   
79344     1340880      2019-04-02              0  train          0   

       time_to_event  
0               15.0  
1              137.0  
2              170.0  
3              110.0  
4              141.0  
...              ... 

In [1]:
import meds_reader
mimic_data = meds_reader.SubjectDatabase("/shared/share_mala/zj2398/mimic/data/meds_v0.6_reader")
print(len(mimic_data))

364627


In [4]:
import pandas as pd
from pathlib import Path
held_out_path = Path("/data/results/zj2398/femr/cumc_0.1/phenotype/tpp/labels/AMI.parquet")
# split = "train.parquet"
sample_parquet = pd.read_parquet(held_out_path)
print(sample_parquet)

        subject_id prediction_time  boolean_value     split  tte_label  \
0         11459411      2019-08-27              0  held_out          0   
1          8624537      2021-10-03              0  held_out          0   
2          5504403      2020-11-29              0  held_out          0   
3          7135352      2020-12-26              0  held_out          0   
4           812724      2022-11-14              0  held_out          0   
...            ...             ...            ...       ...        ...   
148806    15116103      2018-03-01              0     train          0   
148807     3951087      2011-12-08              0     train          0   
148808      540844      2016-10-07              0     train          0   
148809     5203792      2017-10-29              0     train          0   
148810     1340880      2019-04-02              0     train          0   

        time_to_event  
0              1546.0  
1               819.0  
2              1120.0  
3              